# Dice Calculation From Binary Spot Labels

`inputs/mask_hypo.csv` contains the independent manual annotations used as the reference for Dice comparison.
Each cutoff is evaluated against this same fixed mask.
Input provenance is documented in README.md and provenance.json.

# 1. Set Up

Run from this notebook's directory. The six input CSV files contain only
sample ID, spot ID, and `is_hypo` (0 or 1).

In [1]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd
from IPython.display import display
from calculate_dice import read_labels, align_labels, save_results

INPUT_DIR = Path('inputs')
OUTPUT_DIR = Path('results_notebook')
CUTOFFS = [30, 35, 40, 45, 50]
EXAMPLE_SAMPLE = 'UKF248_T_1'
EXAMPLE_CUTOFF = 40
pd.options.display.float_format = '{:.6f}'.format
print({'Python': platform.python_version(), 'NumPy': np.__version__, 'pandas': pd.__version__})

{'Python': '3.11.6', 'NumPy': '1.26.4', 'pandas': '2.1.4'}


# 2. Read the Reference Mask

The same reference mask is used for every cutoff. Both positive and negative
spots are retained.

In [2]:
reference = read_labels(INPUT_DIR / 'mask_hypo.csv')
display(reference.head(8))
sample_sizes = reference.groupby('sample').agg(
    n_spots=('spot_id', 'size'), n_mask_hypo=('is_hypo', 'sum'))
display(sample_sizes)
print('Samples:', reference['sample'].nunique(), '| Total spots:', len(reference))

,sample,spot_id,is_hypo
0,UKF243_T_1,AAACAAGTATCTCCCA-1_#UKF243_T_ST_1,1
1,UKF243_T_1,AAACATTTCCCGGATT-1_#UKF243_T_ST_1,1
2,UKF243_T_1,AAACCGGGTAGGTACC-1_#UKF243_T_ST_1,0
3,UKF243_T_1,AAACCGTTCGTCCAGG-1_#UKF243_T_ST_1,0
4,UKF243_T_1,AAACCTCATGAAGTTG-1_#UKF243_T_ST_1,0
5,UKF243_T_1,AAACGAGACGGTTGAT-1_#UKF243_T_ST_1,1
6,UKF243_T_1,AAACGTGTTCGCCCTA-1_#UKF243_T_ST_1,1
7,UKF243_T_1,AAACTGCTGGCTCCAA-1_#UKF243_T_ST_1,0


,n_spots,n_mask_hypo
sample,,
UKF243_T_1,1844,955
UKF248_T_1,3020,486
UKF255_T_1,1987,932
UKF259_T_2,3546,372
UKF260_T_1,2846,732
UKF275_T_1,3724,324
UKF313_T_2,2677,685
UKF334_T_2,3194,474


Samples: 8 | Total spots: 22838


# 3. Read the Five Hypo Label Tables

These are supplied processed PAN region memberships (`PAN_core`), not raw
abundance thresholds. This calculation reads the existing binary labels;
it does not recalculate thresholds or alter regions.

In [3]:
predictions = {cutoff: read_labels(INPUT_DIR / f'hypo_{cutoff}.csv') for cutoff in CUTOFFS}
display(predictions[EXAMPLE_CUTOFF].head(8))
display(pd.DataFrame([
    {'cutoff_percent': cutoff, 'n_spots': len(frame), 'n_prediction_hypo': int(frame.is_hypo.sum())}
    for cutoff, frame in predictions.items()
]))

,sample,spot_id,is_hypo
0,UKF243_T_1,AAACAAGTATCTCCCA-1_#UKF243_T_ST_1,1
1,UKF243_T_1,AAACATTTCCCGGATT-1_#UKF243_T_ST_1,0
2,UKF243_T_1,AAACCGGGTAGGTACC-1_#UKF243_T_ST_1,0
3,UKF243_T_1,AAACCGTTCGTCCAGG-1_#UKF243_T_ST_1,1
4,UKF243_T_1,AAACCTCATGAAGTTG-1_#UKF243_T_ST_1,0
5,UKF243_T_1,AAACGAGACGGTTGAT-1_#UKF243_T_ST_1,1
6,UKF243_T_1,AAACGTGTTCGCCCTA-1_#UKF243_T_ST_1,1
7,UKF243_T_1,AAACTGCTGGCTCCAA-1_#UKF243_T_ST_1,0


,cutoff_percent,n_spots,n_prediction_hypo
0,30,22838,5982
1,35,22838,5410
2,40,22838,4732
3,45,22838,4165
4,50,22838,3353


# 4. Align and Validate Spots

Align on `(sample, spot_id)`, not row position. Missing, extra, duplicate,
or non-binary rows cause an error. No spots are removed by an inner join.

In [4]:
aligned = {cutoff: align_labels(reference, frame) for cutoff, frame in predictions.items()}
alignment_checks = pd.DataFrame([
    {'cutoff_percent': cutoff, 'n_spots': len(frame), 'n_samples': frame['sample'].nunique(),
     'all_reference_spots_retained': len(frame) == len(reference)}
    for cutoff, frame in aligned.items()
])
display(alignment_checks)
display(aligned[EXAMPLE_CUTOFF].head(8))

,cutoff_percent,n_spots,n_samples,all_reference_spots_retained
0,30,22838,8,True
1,35,22838,8,True
2,40,22838,8,True
3,45,22838,8,True
4,50,22838,8,True


,sample,spot_id,mask_is_hypo,prediction_is_hypo
0,UKF243_T_1,AAACAAGTATCTCCCA-1_#UKF243_T_ST_1,1,1
1,UKF243_T_1,AAACATTTCCCGGATT-1_#UKF243_T_ST_1,1,0
2,UKF243_T_1,AAACCGGGTAGGTACC-1_#UKF243_T_ST_1,0,0
3,UKF243_T_1,AAACCGTTCGTCCAGG-1_#UKF243_T_ST_1,0,1
4,UKF243_T_1,AAACCTCATGAAGTTG-1_#UKF243_T_ST_1,0,0
5,UKF243_T_1,AAACGAGACGGTTGAT-1_#UKF243_T_ST_1,1,1
6,UKF243_T_1,AAACGTGTTCGCCCTA-1_#UKF243_T_ST_1,1,1
7,UKF243_T_1,AAACTGCTGGCTCCAA-1_#UKF243_T_ST_1,0,0


# 5. Calculate Dice for One Sample

Let $A$ be mask-positive spots and $B$ be predicted Hypo-positive spots.

$$\mathrm{Dice}=\frac{2|A\cap B|}{|A|+|B|}
=\frac{2TP}{2TP+FP+FN}.$$

If both positive sets are empty, report NaN. True negatives do not enter Dice.

In [5]:
example = aligned[EXAMPLE_CUTOFF].query('sample == @EXAMPLE_SAMPLE')
a = example['mask_is_hypo'].to_numpy(dtype=bool)
b = example['prediction_is_hypo'].to_numpy(dtype=bool)
tp = int((a & b).sum())
fp = int((~a & b).sum())
fn = int((a & ~b).sum())
tn = int((~a & ~b).sum())
denominator = int(a.sum() + b.sum())
dice = 2 * tp / denominator if denominator else np.nan
display(pd.DataFrame([{'sample': EXAMPLE_SAMPLE, 'cutoff_percent': EXAMPLE_CUTOFF,
    'n_spots': len(a), 'n_mask_hypo': int(a.sum()), 'n_prediction_hypo': int(b.sum()),
    'intersection': tp, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    'denominator': denominator, 'dice': dice}]))
print(f'Dice = 2 * {tp} / ({a.sum()} + {b.sum()}) = {dice:.9f}')

,sample,cutoff_percent,n_spots,n_mask_hypo,n_prediction_hypo,intersection,tp,fp,fn,tn,denominator,dice
0,UKF248_T_1,40,3020,486,411,328,328,83,158,2451,897,0.731327


Dice = 2 * 328 / (486 + 411) = 0.731326644


# 6. Calculate All Samples and Cutoffs

Repeat the same formula independently for each sample and cutoff.

In [6]:
records = []
for cutoff, frame in aligned.items():
    for sample, group in frame.groupby('sample', sort=True):
        a = group['mask_is_hypo'].to_numpy(dtype=bool)
        b = group['prediction_is_hypo'].to_numpy(dtype=bool)
        tp, fp, fn, tn = (int(values.sum()) for values in [a & b, ~a & b, a & ~b, ~a & ~b])
        denominator = int(a.sum() + b.sum())
        records.append({'sample': sample, 'cutoff_percent': cutoff, 'n_spots': len(a),
            'n_mask_hypo': int(a.sum()), 'n_prediction_hypo': int(b.sum()),
            'intersection': tp, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'denominator': denominator, 'dice': 2 * tp / denominator if denominator else np.nan,
            'dice_defined': denominator > 0})
per_sample = pd.DataFrame(records)
display(per_sample)
assert len(per_sample) == reference['sample'].nunique() * len(CUTOFFS)
assert per_sample.dice_defined.all(), 'An empty-positive comparison needs explicit handling.'

,sample,cutoff_percent,n_spots,n_mask_hypo,n_prediction_hypo,intersection,tp,fp,fn,tn,denominator,dice,dice_defined
0,UKF243_T_1,30,1844,955,1029,785,785,244,170,645,1984,0.791331,True
1,UKF248_T_1,30,3020,486,614,388,388,226,98,2308,1100,0.705455,True
2,UKF255_T_1,30,1987,932,1027,812,812,215,120,840,1959,0.828994,True
3,UKF259_T_2,30,3546,372,517,345,345,172,27,3002,889,0.776153,True
4,UKF260_T_1,30,2846,732,888,652,652,236,80,1878,1620,0.804938,True
5,UKF275_T_1,30,3724,324,356,299,299,57,25,3343,680,0.879412,True
6,UKF313_T_2,30,2677,685,1047,613,613,434,72,1558,1732,0.707852,True
7,UKF334_T_2,30,3194,474,504,379,379,125,95,2595,978,0.775051,True
8,UKF243_T_1,35,1844,955,959,750,750,209,205,680,1914,0.783699,True
9,UKF248_T_1,35,3020,486,486,357,357,129,129,2405,972,0.734568,True


# 7. Summarize Across Samples

Give each sample equal weight: mean Dice = sum of sample Dice / number of
defined samples. This is not a pooled-spot Dice. Individual sample peaks
need not coincide with the peak of the mean.

In [7]:
summary = per_sample.groupby('cutoff_percent', as_index=False).agg(
    n_samples=('sample', 'size'), n_defined=('dice', 'count'), mean_dice=('dice', 'mean'),
    sd_dice=('dice', 'std'), min_dice=('dice', 'min'), max_dice=('dice', 'max'))
wide = per_sample.pivot(index='sample', columns='cutoff_percent', values='dice')
wide.loc['Equal_sample_mean'] = wide.mean(axis=0)
display(summary)
display(wide)
ranked = summary.sort_values('mean_dice', ascending=False)
best, runner_up = ranked.iloc[0], ranked.iloc[1]
print(f'Highest mean: cutoff {int(best.cutoff_percent)}%, Dice = {best.mean_dice:.9f}')
print(f'Gap to next-highest mean = {best.mean_dice - runner_up.mean_dice:.9f}')

,cutoff_percent,n_samples,n_defined,mean_dice,sd_dice,min_dice,max_dice
0,30,8,8,0.783648,0.058222,0.705455,0.879412
1,35,8,8,0.793541,0.044873,0.734568,0.866261
2,40,8,8,0.799011,0.046817,0.731327,0.859843
3,45,8,8,0.763414,0.076380,0.611898,0.828841
4,50,8,8,0.674175,0.164083,0.315018,0.813065


cutoff_percent,30,35,40,45,50
sample,,,,,
UKF243_T_1,0.791331,0.783699,0.768061,0.741045,0.645036
UKF248_T_1,0.705455,0.734568,0.731327,0.703529,0.586301
UKF255_T_1,0.828994,0.831142,0.836757,0.820829,0.759494
UKF259_T_2,0.776153,0.782910,0.821078,0.809219,0.762036
UKF260_T_1,0.804938,0.820972,0.834542,0.828841,0.813065
UKF275_T_1,0.879412,0.866261,0.859843,0.827586,0.795883
UKF313_T_2,0.707852,0.739355,0.745004,0.611898,0.315018
UKF334_T_2,0.775051,0.789418,0.795480,0.764361,0.716564
Equal_sample_mean,0.783648,0.793541,0.799011,0.763414,0.674175


Highest mean: cutoff 40%, Dice = 0.799011409
Gap to next-highest mean = 0.005470848


# 8. Export Results

Save per-sample counts, the equally weighted summary, and the wide Dice table.

In [8]:
saved_summary, saved_wide = save_results(per_sample, OUTPUT_DIR)
pd.testing.assert_frame_equal(summary, saved_summary)
pd.testing.assert_frame_equal(wide, saved_wide)
alignment_checks.to_csv(OUTPUT_DIR / 'alignment_checks.csv', index=False)
for path in sorted(OUTPUT_DIR.glob('*.csv')):
    print(path)

results_notebook/alignment_checks.csv
results_notebook/dice_per_sample.csv
results_notebook/dice_summary.csv
results_notebook/dice_wide.csv
